# 03 - Modeling: Logistic Regression, Random Forest, XGBoost

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, classification_report, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import build_pipeline

DATA_PATH = os.path.join('..', 'data', 'raw', 'fake_job_postings.csv')
df = pd.read_csv(DATA_PATH)
X = df.drop(columns=['fraudulent'])
y = df['fraudulent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
results = {}
for model_type in ['logistic_regression', 'random_forest', 'xgboost']:
    pipeline = build_pipeline(model_type, calibrate=True)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    metrics = {
        'fraud_f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'average_precision': average_precision_score(y_test, y_proba),
    }
    results[model_type] = {'pipeline': pipeline, 'metrics': metrics}
    print(f'--- {model_type} ---')
    print(classification_report(y_test, y_pred, target_names=['Real', 'Fake'], zero_division=0))
    print('ROC-AUC:', metrics['roc_auc'])
    print('PR-AUC:', metrics['average_precision'])


In [ ]:
best_model_type = max(
    results,
    key=lambda name: (results[name]['metrics']['fraud_f1'], results[name]['metrics']['average_precision'])
)
best_pipeline = results[best_model_type]['pipeline']
print('Best model:', best_model_type, results[best_model_type]['metrics'])

import joblib
os.makedirs('../models', exist_ok=True)
joblib.dump(best_pipeline, '../models/model.pkl')
print('Saved calibrated model to ../models/model.pkl')


In [ ]:
y_pred = best_pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real','Fake'], yticklabels=['Real','Fake'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title(f'Confusion Matrix - {best_model_type}')
plt.savefig('../reports/figures/confusion_matrix.png', bbox_inches='tight')
plt.show()